# IMPORTACIONES Y DATOS

In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

X = pd.read_csv("training_features.csv")
y = pd.read_csv("training_target.csv")

target = y.columns[0]

# IDENTIFICACIÓN DE COLUMNAS

In [4]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object", "string"]).columns

print("Numéricas:", len(num_cols))
print("Categóricas:", len(cat_cols))

Numéricas: 36
Categóricas: 43


In [7]:
# Columnas ordinales
ordinal_cols = [
    "Exter Qual",
    "Exter Cond",
    "Bsmt Qual",
    "Bsmt Cond",
    "Heating QC",
    "Kitchen Qual",
    "Fireplace Qu",
    "Garage Qual",
    "Garage Cond",
    "Pool QC"
]

# Orden definido
quality_order = ["None", "Po", "Fa", "TA", "Gd", "Ex"]

# Columnas nominales
nominal_cols = [col for col in cat_cols if col not in ordinal_cols]

# PIPELINES INDIVIDUALES

In [8]:
# Numericas
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Ordinales
ordinal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
    ("encoder", OrdinalEncoder(categories=[quality_order]*len(ordinal_cols)))
])

# Nominales
nominal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# COLUMNTRANSFORMER

In [9]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, num_cols),
    ("ord", ordinal_pipeline, ordinal_cols),
    ("nom", nominal_pipeline, nominal_cols)
])

# SEPARACIÓN TRAIN / VALIDATION

In [10]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y[target],
    test_size=0.2,
    random_state=42
)

# AJUSTAR PREPROCESADOR

In [11]:
preprocessor.fit(X_train)

X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)

print("Shape train procesado:", X_train_processed.shape)
print("Shape val procesado:", X_val_processed.shape)

Shape train procesado: (1875, 267)
Shape val procesado: (469, 267)
